In [1]:
import fft_poisson_solver_opt
import fft_poisson_solver
import numpy as np
from matplotlib import pyplot as plt
import imageio
from scipy import sparse
import os
import tvl1_poisson_solver
import torch

pyFFTW will use up to 64 threads.


In [ ]:
file_name = "test_shift"
folder_name = "../results/grayscale_example/image_registration/"
result_dir = "../results/grayscale_example/reconstruction_l1/"
os.makedirs(result_dir, exist_ok=True)

In [3]:
sparse_matrix_x = np.load(folder_name+file_name+'_x.npy')
sparse_matrix_y = np.load(folder_name+file_name+'_y_warp.npy')

print(sparse_matrix_x.max())
print(sparse_matrix_x.min())
print(sparse_matrix_y.max())
print(sparse_matrix_y.min())
print(np.abs(sparse_matrix_x).max())
print(np.abs(sparse_matrix_y).max())
print(np.abs(sparse_matrix_x).min())
print(np.abs(sparse_matrix_y).min())

1.0
-1.0666666666666667
1.0
-1.0476190476190477
1.0666666666666667
1.0476190476190477
0.0
0.0


In [4]:
grad_x_torch = torch.from_numpy(sparse_matrix_x)
grad_y_torch = torch.from_numpy(sparse_matrix_y)
with torch.no_grad(): # Disable autograd to save huge memory and speed up
    reconstructed_gpu, p_gpu = tvl1_poisson_solver.tv_l1_reconstruction_cuda(
        grad_x_torch, 
        grad_y_torch, 
        lambda_tv=1.0, 
        n_iters=5000, 
        device='cuda:1'
    )
event_reconstructed = reconstructed_gpu.cpu().numpy()
print(f"Shape: {event_reconstructed.shape}")
print(f"Min: {np.min(event_reconstructed)}")
print(f"Max: {np.max(event_reconstructed)}")

Shape: (4000, 4000)
Min: -6.989985466003418
Max: 4.007730007171631


In [5]:
# event_reconstructed = -fft_poisson_solver.poisson_solver_with_brightness(sparse_matrix_x, sparse_matrix_y, u0=-1, lam=5e-6)
# print(f"Shape: {event_reconstructed.shape}")
# print(f"Min: {np.min(event_reconstructed)}")
# print(f"Max: {np.max(event_reconstructed)}")

In [ ]:
# Normalize to [0,1] range
event_reconstructed = (event_reconstructed - np.min(event_reconstructed)) / (np.max(event_reconstructed) - np.min(event_reconstructed))
# # Exponentiate the reconstructed values to linearize the image
event_reconstructed = np.exp(event_reconstructed)
# # Apply gamma correction with gamma=2.2
event_reconstructed = event_reconstructed ** (1/2.2)
event_reconstructed = (event_reconstructed - np.min(event_reconstructed)) / (np.max(event_reconstructed) - np.min(event_reconstructed))
# np.save(result_dir+file_name+"_reconstructed.npy", event_reconstructed)
imageio.imwrite(result_dir+file_name+"_reconstructed.png", (event_reconstructed * 255.0).astype(np.uint8))

# Auto HDR based on histogram analysis
hist, bins = np.histogram(event_reconstructed.flatten(), bins=65536)
cumsum = np.cumsum(hist)
cumsum = cumsum / cumsum[-1] # Normalize to [0,1]

# Find points for histogram equalization 
low_percentile = 15/65535  # Bottom 0.000015%
high_percentile = 65000/65535 # Top 0.000015%

# Find the intensity values at these percentiles
low_value = bins[np.where(cumsum > low_percentile)[0][0]]
high_value = bins[np.where(cumsum > high_percentile)[0][0]]

# Apply exposure adjustment based on histogram analysis
exposure_factor = 1.0 / (high_value - low_value)
event_reconstructed_hdr = (event_reconstructed - low_value) * exposure_factor

# Clip to ensure values stay in [0,1] range after exposure increase
event_reconstructed_hdr = np.clip(event_reconstructed_hdr, 0, 1)
# np.save(result_dir+file_name+"_reconstructed.npy", event_reconstructed_hdr)
# Convert to 0-255 range
event_reconstructed_hdr = (event_reconstructed_hdr * 255).astype(np.uint8)
# Save using imageio

# imageio.imwrite(result_dir+file_name+"_reconstructed_downsampled_8x.png", event_reconstructed[::8, ::8])
imageio.imwrite(result_dir+file_name+"_reconstructed_hdr.png", event_reconstructed_hdr)


In [7]:
import torch
import numpy as np
import imageio
from tqdm import tqdm

# Check if GPU is available
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def auto_hdr_vectorized(patch_batch, bins=65536):
    """
    Vectorized Auto HDR for a batch of patches [Batch, Height, Width].
    Processes B patches in parallel.
    """
    # Flatten spatial dims: [B, H, W] -> [B, N]
    B, H, W = patch_batch.shape
    flat_patches = patch_batch.reshape(B, -1)
    
    # 1. Compute Min/Max per patch -> Shape [B, 1]
    min_vals = flat_patches.min(dim=1, keepdim=True).values
    max_vals = flat_patches.max(dim=1, keepdim=True).values
    ranges = max_vals - min_vals
    
    # Avoid division by zero
    ranges = ranges + 1e-8
    
    # 2. Normalize and Quantize for Histogram
    # [B, N]
    normalized = (flat_patches - min_vals) / ranges
    scaled_indices = (normalized * (bins - 1)).long()
    
    # 3. Vectorized Histogram (The Offset Trick)
    # Offset indices so each batch falls into its own bucket range
    offsets = (torch.arange(B, device=patch_batch.device) * bins).unsqueeze(1)
    flat_indices = (scaled_indices + offsets).view(-1)
    
    # Compute one giant bincount
    counts = torch.bincount(flat_indices, minlength=B*bins).float()
    
    # Reshape back to [B, bins]
    hists = counts.view(B, bins)
    
    # 4. CDF and Percentiles
    cumsum = torch.cumsum(hists, dim=1)
    total_counts = cumsum[:, -1:].clamp(min=1.0) 
    cdf = cumsum / total_counts
    
    # Percentile Targets
    low_p = 15.0 / 65535.0
    high_p = 65000.0 / 65535.0
    # low_p = 15.0 / 255.0
    # high_p = 235.0 / 255.0
    
    # --- FIX START ---
    # Create target tensors of shape [B, 1] so searchsorted works per-row
    low_targets = torch.full((B, 1), low_p, device=patch_batch.device)
    high_targets = torch.full((B, 1), high_p, device=patch_batch.device)
    
    # Output shape will be [B, 1]
    low_idx = torch.searchsorted(cdf, low_targets).float()
    high_idx = torch.searchsorted(cdf, high_targets).float()
    # --- FIX END ---
    
    # 5. Map back to intensity values
    bin_width = ranges / bins # [B, 1]
    
    # Note: low_idx is already [B, 1], so we DO NOT need unsqueeze(1) here anymore
    low_val = min_vals + low_idx * bin_width
    high_val = min_vals + high_idx * bin_width
    
    # 6. Apply Exposure Correction
    val_diff = high_val - low_val
    
    # Check for low dynamic range ( < 0.25 )
    # mask_diff = high_val - flat_patches
    small_range_mask = (ranges < 0.15) & (min_vals > 0.5) # [B, 1]
    
    exposure_factor = 1.0 / (val_diff + 1e-8)
    
    # Apply calculation (Broadcasting [B, N] - [B, 1])
    result = (flat_patches - low_val) * exposure_factor
    
    # Handle the small range case (saturate to high_val)
    result = torch.where(small_range_mask, 1.0, result)
    
    # Reshape back to [B, H, W]
    return result.view(B, H, W)

# --- Main Execution ---

# 1. Setup Data on GPU
patch_size = 100
step = 2
h, w = event_reconstructed.shape
pad_amt = patch_size // 2

# Move original image to GPU immediately to avoid repeated transfers
# Pad on GPU
img_tensor = torch.from_numpy(event_reconstructed / 255.0).float().to(device)
img_tensor = img_tensor / (torch.abs(img_tensor).max() + 1e-8)
img_padded = torch.nn.functional.pad(img_tensor, (pad_amt, pad_amt, pad_amt, pad_amt), mode='constant', value=0)

# Accumulation Buffers on GPU
# We accumulate float results directly on VRAM
output_accumulator = torch.zeros_like(img_padded)
weight_accumulator = torch.zeros_like(img_padded)

# Pre-calculate coordinates
# Original shape coords
rows = list(range(0, h, step))
cols = list(range(0, w, step))
patch_coords = [(r, c) for r in rows for c in cols]

# 2. Process in Batches
batch_size = 1024  # Safe size for VRAM. You can try 1024 if 24GB allows.

print(f"Total patches: {len(patch_coords)}")

# Create a dummy ones tensor for weight accumulation
ones_patch = torch.ones((patch_size, patch_size), device=device)

for batch_start in tqdm(range(0, len(patch_coords), batch_size), desc="GPU Processing"):
    batch_end = min(batch_start + batch_size, len(patch_coords))
    current_coords = patch_coords[batch_start:batch_end]
    
    if not current_coords:
        continue

    # Extract Patches on GPU
    # List comprehension creates a list of views/tensors
    # torch.stack packs them into a single tensor [B, 320, 320]
    # This is extremely fast because data is already on device
    batch_patches = torch.stack([
        img_padded[r:r+patch_size, c:c+patch_size] 
        for r, c in current_coords
    ])
    
    # Run Vectorized HDR (No loop inside here)
    processed_batch = auto_hdr_vectorized(batch_patches)
    
    # Accumulate back to main image
    # We loop here because scatter_add or fancy indexing for overlapping blocks 
    # is complex to implement efficiently. Since B is small (512), this loop is fast enough.
    for idx, (r, c) in enumerate(current_coords):
        output_accumulator[r:r+patch_size, c:c+patch_size] += processed_batch[idx]
        weight_accumulator[r:r+patch_size, c:c+patch_size] += ones_patch

# 3. Normalize and Save
# Avoid division by zero
weight_accumulator[weight_accumulator == 0] = 1.0
final_output = output_accumulator / weight_accumulator

# Crop padding
final_output = final_output[pad_amt:pad_amt+h, pad_amt:pad_amt+w]

# Clip and Convert
final_output = torch.clamp(final_output, 0, 1)
result_uint8 = (final_output * 255).byte().cpu().numpy()

imageio.imwrite(result_dir+file_name+"_reconstructed_auto_hdr_optimized.png", result_uint8)

Using device: cuda:1
Total patches: 4000000


GPU Processing: 100%|██████████| 3907/3907 [01:44<00:00, 37.53it/s]
